# qAIR-vNext v43 -- Local Training (cell by cell)

Runs the same steps as `python main.py --mode train`
(`training/runner.py::run_training`), but split into separate cells so
each stage -- dataset/cache build, a single sample, a single batch, the
model, the trainer, the actual training loop, a quick eval -- can be
inspected on its own before moving to the next one.

This is for **local** use (your PC or Mac), not Colab -- no Drive
mount, no repo clone, no tensorflow uninstall step. It assumes you've
already run, in this repo's root:

```
python -m venv venv
.\venv\Scripts\Activate.ps1      # or: source venv/bin/activate on Mac
pip install torch
pip install -r requirements.txt
```

and are running this notebook with that venv's kernel selected.

Uses its own checkpoint name (`qair_v43_*`) so it won't collide with
checkpoints from `main.py --mode train` (currently `qair_v42_*`) or
the ablation suite (`A1_baseline_*`, etc.) -- change `RUN_NAME` below
if you want this run to resume from/overwrite a different name.

## 0. Make sure we're running from the repo root

This notebook lives in `notebooks/`, but every `from models...` /
`from training...` import and every relative path in `config.py`
(`./cache`, `./ckpt`) assumes the working directory is the repo root.
Jupyter/VS Code often default the kernel's cwd to the notebook's own
folder -- fix it here if so, once, before anything else runs.

In [ ]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working directory:", os.getcwd())
assert os.path.exists("config.py"), (
    "Not at the repo root -- adjust the os.chdir(...) above to point "
    "here manually, e.g. os.chdir(r'D:\\WORK\\Research\\qAIR-CSE499B')"
)

## 0b. Force offline mode for Hugging Face downloads

The Qwen2.5-0.5B-Instruct model and the MiniLM encoder are already
fully downloaded and cached locally (verified: ~953MB in
`~/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B-Instruct`).
Without this, `huggingface_hub` still makes a network round-trip on
every load to check whether the cached files are still current --
and on this machine that request appears to hang indefinitely
(confirmed: the kernel process sits at 0% CPU, i.e. blocked on I/O,
not computing) rather than failing fast, likely a firewall/proxy/AV
silently swallowing the request.

Setting `HF_HUB_OFFLINE=1` skips that network check entirely and
loads straight from the local cache. **If you ever need to download a
*new* model for the first time on this machine** (or run this
notebook somewhere nothing is cached yet, e.g. a fresh Mac), comment
this cell out first -- offline mode will fail fast with a clear "file
not found" error instead of downloading, which is what you want.

In [ ]:
import os

os.environ["HF_HUB_OFFLINE"] = "1"

print("HF_HUB_OFFLINE:", os.environ["HF_HUB_OFFLINE"])

## 1. Imports, seed, device

`resolve_device()` picks cuda > mps > cpu -- confirm here which one
your machine actually got before committing to a full run.

In [ ]:
from functools import partial

import torch
from torch.utils.data import DataLoader

from config import (
    CACHE_DIR,
    CKPT_DIR,
    EPOCHS,
    PATIENCE,
    PERSISTENT_STEPS,
    N_QUBITS,
    BATCH_SIZE,
    WEIGHT_DECAY,
    resolve_device,
)

from training.seed import set_seed
from training.dataset import QAIRDataset, collate_fn, DIM
from training.train import Trainer
from training.checkpoint import load_or_resume
from training.evaluate import evaluate
from models.full_model import QAIRvNext

set_seed(42)

device = resolve_device()
print("Device:", device)

## 2. Configuration for this run

Same parameters `run_training()` takes. Set `TRAIN_SAMPLES`/`VAL_SAMPLES`
to a small number (e.g. 40/20) for a fast smoke test of the whole
pipeline before committing to the full dataset -- `None` means "use
everything".

In [ ]:
RUN_NAME = "qair_v43"

TRAIN_SAMPLES = None   # e.g. 40 for a quick smoke test
VAL_SAMPLES = None     # e.g. 20 for a quick smoke test

epochs = EPOCHS
persistent_steps = PERSISTENT_STEPS
n_qubits = N_QUBITS

print(f"epochs={epochs}  persistent_steps={persistent_steps}  n_qubits={n_qubits}")
print(f"batch_size={BATCH_SIZE}  weight_decay={WEIGHT_DECAY}  patience={PATIENCE}")

## 3. Build (or load cached) datasets

First run builds `cache/arc_train.pt` / `cache/arc_validation.pt` from
scratch -- generates one hypothesis per answer option via the LLM for
every ARC question, then embeds everything. This is the slow, one-time
step; every run after this reuses the cache instantly. Progress bars
will show which phase (train vs. validation, and generation vs.
resume) is running.

In [ ]:
train_ds = QAIRDataset(split="train", max_samples=TRAIN_SAMPLES, cache_dir=CACHE_DIR)
val_ds = QAIRDataset(split="validation", max_samples=VAL_SAMPLES, cache_dir=CACHE_DIR)

print(f"Train samples: {len(train_ds)}")
print(f"Val samples:   {len(val_ds)}")

## 3b. Inspect a single sample

Before training on it -- confirm the hypotheses look reasonable and
the embedding shapes match `config.EMBEDDING_DIM`.

In [ ]:
sample = train_ds[0]

print("Question:", sample["question"])
print()
print("Options:", sample["options"])
print()
print("Generated hypotheses:")
for i, h in enumerate(sample["hypotheses"]):
    print(f"  {i}. {h}")
print()
print("H shape:", sample["H"].shape)
print("O shape:", sample["O"].shape)
print("Correct answer index (y):", sample["y"])

## 4. DataLoaders

In [ ]:
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=partial(collate_fn, shuffle_options=True),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=partial(collate_fn, shuffle_options=False),
)

print(f"Train batches per epoch: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 4b. Inspect a single batch

In [ ]:
batch = next(iter(train_loader))

for k, v in batch.items():
    print(f"{k:8s} shape={tuple(v.shape)} dtype={v.dtype}")

## 5. Build the model

`use_quantum=True, use_validator=True` matches `run_training()`'s
defaults (the full model, not an ablation config).

In [ ]:
model = QAIRvNext(
    dim=DIM,
    use_quantum=True,
    use_validator=True,
    persistent_steps=persistent_steps,
    n_qubits=n_qubits,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {n_params:,}")
print(f"Trainable parameters: {n_trainable:,}")
print(f"Model device: {next(model.parameters()).device}")

## 6. Trainer + checkpoint resume

If `ckpt/{RUN_NAME}_latest.pt` already exists (from a previous run of
this notebook), this resumes from it instead of starting over.

In [ ]:
import os

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    ckpt_dir=CKPT_DIR,
    name=RUN_NAME,
    weight_decay=WEIGHT_DECAY,
)

latest_ckpt = os.path.join(CKPT_DIR, f"{RUN_NAME}_latest.pt")
print("Checkpoint path:", latest_ckpt)
print("Exists:", os.path.exists(latest_ckpt))

start_epoch, best_acc, _ = load_or_resume(trainer, CKPT_DIR, RUN_NAME, epochs)
print(f"Starting from epoch {start_epoch} / {epochs}  (best_acc so far: {best_acc:.4f})")

## 7. Train

This is the long-running cell -- per-epoch progress bars, validation
metrics, and best/latest checkpoint saves print as they happen.
Interrupting the kernel here is safe: the trainer saves a `_latest.pt`
checkpoint every epoch, so re-running this cell (after re-running cell
6 to rebuild the `Trainer`) resumes from the last completed epoch
instead of restarting.

In [ ]:
history = trainer.train(
    epochs=epochs,
    start_epoch=start_epoch,
    best_acc=best_acc,
    patience=PATIENCE,
)

## 8. Save the run config alongside the checkpoint

So a later `evaluation/sample_inference.py`-style load can't silently
mismatch `n_qubits`/`persistent_steps` against what this checkpoint
was actually trained with.

In [ ]:
config_path = os.path.join(CKPT_DIR, f"{RUN_NAME}_config.pt")

torch.save(
    {
        "use_quantum": True,
        "use_validator": True,
        "persistent_steps": persistent_steps,
        "n_qubits": n_qubits,
    },
    config_path,
)

print(f"[CONFIG SAVED] {config_path}")

## 9. Quick validation-set evaluation

Same `evaluate()` used during training, run once more standalone on
the final model state.

In [ ]:
metrics = evaluate(model, val_loader, device)

for k, v in metrics.items():
    print(f"{k:15s}: {v:.4f}")

## 10. Helpers for ablation results

Small local versions of the print/report helpers `qair_v42_colab.ipynb`
uses, scoped to what the sections below need.

In [ ]:
def checkpoint_status(path, label="Checkpoint"):
    print("=" * 60)
    print(label)
    print("=" * 60)
    print("Path:", path)
    print("Exists:", os.path.exists(path))
    if not os.path.exists(path):
        print("[NO CHECKPOINT FOUND] Will train from scratch.")
        return None
    try:
        ckpt = torch.load(path, map_location="cpu")
        epoch = ckpt.get("epoch", "UNKNOWN")
        print(f"epoch = {epoch}")
        if isinstance(epoch, int):
            print(f"start_epoch = {epoch + 1}")
        return ckpt
    except Exception as e:
        print("[CHECKPOINT INCOMPATIBLE / FAILED TO LOAD]")
        print(e)
        return None


def save_report(metrics, export_dir, filename="evaluation.txt"):
    os.makedirs(export_dir, exist_ok=True)
    report_path = os.path.join(export_dir, filename)
    with open(report_path, "w") as f:
        for k, v in metrics.items():
            f.write(f"{k}: {v}\n")
    print("Saved to:", report_path)
    return report_path


def list_exports(export_dir):
    print("=" * 60)
    print("EXPORTED FILES")
    print("=" * 60)
    for f in sorted(os.listdir(export_dir)):
        print(f)


EXPORT_DIR = "./exports"
os.makedirs(EXPORT_DIR, exist_ok=True)

## 11. Check existing ablation checkpoints

All 7 configs in `training/ablations.py::ABLATIONS` -- including the
two parameter-matched classical-control configs (`A1c_...`, `A3c_...`)
added for the quantum-vs-classical comparison. Resume support works
the same way as the single-model training above: any config with a
`_latest.pt` checkpoint picks up where it left off instead of
restarting.

In [ ]:
from training.ablations import run_ablation_suite, ABLATIONS

for name in ABLATIONS.keys():
    print()
    checkpoint_status(os.path.join(CKPT_DIR, f"{name}_latest.pt"), label=name)

## 12. Run the full ablation suite (all 7 configs)

Trains/resumes, in order: `A1_baseline`, `A1b_quantum_only`,
`A1c_classical_control_only`, `A2_validator`, `A3_persistent`,
`A3c_classical_control_persistent`, `A4_full_hybrid`. Reuses the same
cache built in section 3 -- no LLM regeneration needed. This is the
longest-running cell in the notebook (up to 7x a single training run);
each config checkpoints every epoch the same way section 7 does, so
interrupting and re-running this cell resumes rather than restarting.

Watch for the `[QuantumEvolutionLayer]`/`[ClassicalControlLayer]`
parameter-count prints when each config's model is constructed -- that's
the runtime verification that the two backends are actually
parameter-matched, not just a hand calculation.

In [ ]:
try:
    ablation_results = run_ablation_suite(
        cache_dir=CACHE_DIR,
        ckpt_dir=CKPT_DIR,
        epochs=epochs,
        patience=PATIENCE,
        n_qubits=n_qubits,
    )
    print("\n" + "=" * 60)
    print("ABLATION RESULTS")
    print("=" * 60)
    for name, r in ablation_results.items():
        print(f"\n{name}")
        print(f"  config        : {r['config']}")
        print(f"  best_acc      : {r['best_acc']:.4f}")
        print(f"  final_val_acc : {r.get('final_val_acc')}")
except Exception as e:
    print("\n[ABLATION FAILED]")
    print(e)

## 13. Load the full model for evaluation

Targeting `A4_full_hybrid` specifically -- quantum backend + validator
+ persistent_steps=5, the most complete config in the grid. Change
`BEST_CONFIG_NAME` below to inspect a different config instead (e.g.
`A1c_classical_control_only` to compare against the quantum version).

In [ ]:
from evaluation.sample_inference import load_ablation_model, run_sample_evaluation

BEST_CONFIG_NAME = "A4_full_hybrid"   # full model: quantum + validator + persistent_steps=5

eval_model = load_ablation_model(
    ckpt_dir=CKPT_DIR,
    name=BEST_CONFIG_NAME,
    cfg=ABLATIONS[BEST_CONFIG_NAME],
    device=device,
    n_qubits=n_qubits,
)

## 14. Full validation-set evaluation

Reuses `val_ds` from section 3. `shuffle_options=False` here (matching
section 9's reasoning) so these metrics are reproducible run to run.

In [ ]:
eval_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=partial(collate_fn, shuffle_options=False),
)

eval_metrics = evaluate(eval_model, eval_loader, device)
print(eval_metrics)

report_path = save_report(eval_metrics, EXPORT_DIR, filename=f"{BEST_CONFIG_NAME}_evaluation.txt")

## 15. Sample-question sanity check

Runs the 10 hand-written questions in `evaluation/sample_inference.py`
through the loaded model -- a quick, human-readable check independent
of the ARC validation set.

In [ ]:
sample_acc = run_sample_evaluation(model=eval_model, device=device)
print(f"\nSample Accuracy ({BEST_CONFIG_NAME}) = {sample_acc:.4f}")

## 16. Grab one sample for visualization

In [ ]:
viz_loader = DataLoader(val_ds, batch_size=1, shuffle=True, collate_fn=collate_fn)
viz_batch = next(iter(viz_loader))

viz_H = viz_batch["H"].to(device)
viz_O = viz_batch["O"].to(device)

with torch.no_grad():
    viz_out = eval_model(viz_H, viz_O)

## 17. Visualizations

In [ ]:
from visualization.attention_maps import plot_attention_map

attention = viz_out["attention"]
if attention.dim() == 4:
    attention = attention[0, 0]
elif attention.dim() == 3:
    attention = attention[0]

plot_attention_map(attention, save_path=f"{EXPORT_DIR}/{BEST_CONFIG_NAME}_attention_map.png")

In [ ]:
from visualization.energy_maps import plot_energy_map
plot_energy_map(viz_out["answer_energy"][0], save_path=f"{EXPORT_DIR}/{BEST_CONFIG_NAME}_energy_map.png")

In [ ]:
import matplotlib.pyplot as plt

collapse = viz_out["collapse_probs"][0].detach().cpu().numpy()
plt.figure(figsize=(6, 4))
plt.bar(range(len(collapse)), collapse)
plt.xlabel("Hypothesis")
plt.ylabel("Probability")
plt.title("Collapse Distribution")
plt.savefig(f"{EXPORT_DIR}/{BEST_CONFIG_NAME}_collapse_probs.png", bbox_inches="tight")
plt.show()

In [ ]:
from visualization.trajectory import plot_trajectory
plot_trajectory(viz_out["trajectory"], save_path=f"{EXPORT_DIR}/{BEST_CONFIG_NAME}_trajectory.png")

In [ ]:
validator_out = viz_out["validator"]
if validator_out is not None:
    scores = [
        validator_out["causal"][0].mean().item(),
        validator_out["diversity"][0].mean().item(),
        validator_out["specificity"][0].mean().item(),
        validator_out["relevance"][0].mean().item(),
    ]
    labels = ["causal", "diversity", "specificity", "relevance"]
    plt.figure(figsize=(7, 4))
    plt.bar(labels, scores)
    plt.title("Validator Scores")
    plt.savefig(f"{EXPORT_DIR}/{BEST_CONFIG_NAME}_validator.png", bbox_inches="tight")
    plt.show()
else:
    print(f"{BEST_CONFIG_NAME} has use_validator=False -- no validator scores to plot.")

## 18. List exported files

In [ ]:
list_exports(EXPORT_DIR)